In [ ]:
# # 05a. Monte Carlo uncertainty analysis, server run
# 
# This notebook combines the server-side parts of the original uncertainty workflow. It is intended for the high-memory server run.

from pathlib import Path
import gc
import os
import time
import warnings

import h5py
import numpy as np
import pandas as pd
import scipy.io
import scipy.linalg

warnings.filterwarnings("ignore")


def find_submission_dir(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start] + list(start.parents):
        if candidate.name == "submission_code":
            return candidate
        if (candidate / "submission_code").is_dir():
            return candidate / "submission_code"
    if start.name == "5_uncertainty_analysis":
        return start.parent
    return start


SUBMISSION_DIR = find_submission_dir()
NOTEBOOK_DIR = SUBMISSION_DIR / "5_uncertainty_analysis"
INPUT_DIR = SUBMISSION_DIR / "input"
OUTPUT_DIR = NOTEBOOK_DIR / "server_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Expected submission package input layout.
IO_DIR = INPUT_DIR / "io"
HOUSEHOLD_DIR = INPUT_DIR / "household_expenditure"
CF_DIR = INPUT_DIR / "characterization_factors"
REFERENCE_DIR = INPUT_DIR / "reference"

FILES = {
    "sdg": IO_DIR / "99_SDG_ind_1990_2029.mat",
    "use_table": IO_DIR / "UT2023.mat",
    "final_demand": IO_DIR / "FD2023.mat",
    "target": HOUSEHOLD_DIR / "Target_2023.npz",
    "population": HOUSEHOLD_DIR / "Population_by_IncomeGroup.csv",
    "country_mapping": HOUSEHOLD_DIR / "GLORIA_Country_Mapping.csv",
    "direct_lucf": HOUSEHOLD_DIR / "fp_direct_LUCF_2023.npy",
    "cf": CF_DIR / "GLORIA_Chaudhary_CF_long.csv",
    "sector_class": REFERENCE_DIR / "sector_class.xlsx",
}

G, S, B = 164, 120, 201
R = G * S
YEAR = 2023
YEAR_INDEX = YEAR - 1990
N_ITER = 10000
SEED = 42
EPSILON = 1e-9
AREA_TO_M2 = 1e7
LU_NAMES = [
    "Annual_crops",
    "Permanent_crops",
    "Pasture",
    "Intensive_forestry",
    "Extensive_forestry",
    "Urban",
]
r_index = np.arange(R) // S

t_wall = time.time()


def log(message):
    print(f"[{(time.time() - t_wall) / 60:6.1f} min] {message}", flush=True)


def require_files(paths):
    missing = [str(p) for p in paths if not Path(p).exists()]
    if missing:
        raise FileNotFoundError("Missing required input files:\n  " + "\n  ".join(missing))


def load_hdf_matrix(path, candidate_keys):
    with h5py.File(path, "r") as hf:
        keys = list(hf.keys())
        key = next((k for k in candidate_keys if k in hf), keys[0])
        return hf[key][:].astype(np.float64)


def compute_gini(fp, pop):
    fp = np.asarray(fp, float).ravel()
    pop = np.asarray(pop, float).ravel()
    mask = pop > 0
    fp, pop = fp[mask], pop[mask]
    idx = np.argsort(fp / pop)
    fp, pop = fp[idx], pop[idx]
    lx = np.concatenate([[0.0], np.cumsum(pop)]) / pop.sum()
    ly = np.concatenate([[0.0], np.cumsum(fp)]) / fp.sum()
    return float(1 - 2 * np.trapz(ly, lx))


def ci(values):
    values = np.asarray(values, float)
    return float(np.median(values)), float(np.percentile(values, 2.5)), float(np.percentile(values, 97.5))


require_files(FILES.values())
print("Submission directory:", SUBMISSION_DIR)
print("Server outputs:", OUTPUT_DIR)

# Part 1. Build satellite account and independent Monte Carlo factors.
log("Loading land-use satellite account")
mat = scipy.io.loadmat(FILES["sdg"])
SDG = mat["SDG"]
LU = np.maximum(SDG[27:33, :, YEAR_INDEX].copy(), 0.0)
assert LU.shape == (6, R), f"Unexpected land-use shape: {LU.shape}"
del mat, SDG
gc.collect()

log("Loading characterization factors")
cf_df = pd.read_csv(FILES["cf"])
cf_combined = np.zeros((G, 6), dtype=np.float64)
sigma_matrix = np.zeros((G, 6), dtype=np.float64)
for lu_idx, lu_name in enumerate(LU_NAMES):
    sub = cf_df[cf_df["LU_type"] == lu_name].sort_values("GLORIA_ID")
    assert len(sub) == G, f"{lu_name}: expected {G} rows, got {len(sub)}"
    cf_combined[:, lu_idx] = sub["CF_median_PDF_per_m2"].fillna(0).values
    sigma_matrix[:, lu_idx] = sub["sigma_log"].fillna(0).values

cf_sector = cf_combined[r_index]
sat_by_lu = (LU * cf_sector.T) * AREA_TO_M2
sat_base = sat_by_lu.sum(axis=0)

direct = np.load(FILES["direct_lucf"])
print(f"Satellite total PDF: {sat_base.sum():.6e}")
print(f"Direct LUCF check  : {direct.sum():.6e}")
print(f"Absolute difference: {abs(sat_base.sum() - direct.sum()):.3e}")

log("Generating independent Monte Carlo factors")
rng = np.random.default_rng(SEED)
z = rng.standard_normal((N_ITER, G, 6))
mc_factors = np.exp(sigma_matrix[None, :, :] * z).astype(np.float32)

np.save(OUTPUT_DIR / "sat_by_landuse_2023.npy", sat_by_lu.astype(np.float32))
np.save(OUTPUT_DIR / "cf_sigma_landuse.npy", sigma_matrix.astype(np.float32))
np.save(OUTPUT_DIR / "mc_factors_independent.npy", mc_factors)
print("Saved satellite account, sigma matrix, and independent MC factors.")

# Part 2. Build I-A and factorize once.
log("Loading use table Z")
Z = load_hdf_matrix(FILES["use_table"], ["UT2023", "Z", "Z2023", "use_table"])
assert Z.shape == (R, R), f"Unexpected use-table shape: {Z.shape}"

log("Loading final demand FD and computing total output")
FD = load_hdf_matrix(FILES["final_demand"], ["FD", "FD2023", "final_demand"])
if FD.shape[0] != R:
    FD = FD.T
assert FD.shape[0] == R, f"Unexpected final-demand shape: {FD.shape}"

x = Z.sum(axis=1) + FD.sum(axis=1)
x_safe = np.where(x > 0, x, 1.0)
del FD
gc.collect()

log("Building I-A in Fortran order for in-place LU factorization")
ImA = np.empty(Z.shape, dtype=np.float64, order="F")
np.divide(Z, x_safe[None, :], out=ImA)
np.nan_to_num(ImA, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
del Z
gc.collect()
np.negative(ImA, out=ImA)
ImA.flat[:: R + 1] += (1.0 + EPSILON)

log("LU factorization")
lu, piv = scipy.linalg.lu_factor(ImA, overwrite_a=True)
del ImA
gc.collect()

q = np.nan_to_num(sat_base / x_safe, nan=0.0, posinf=0.0, neginf=0.0)
q[x <= 0] = 0.0
print("Leontief system is ready.")

# Part 3. Compute baseline tensor by producing GLORIA sector.
log("Loading household final-demand target")
npz = np.load(FILES["target"])
Target = np.nan_to_num(npz["Target"].astype(np.float32), nan=0.0)
del npz
gc.collect()
assert Target.shape == (R, G * B), f"Unexpected Target shape: {Target.shape}"

fp_by_gloria = np.empty((R, G, B), dtype=np.float32)
for j in range(G):
    T_j = np.array(Target[:, j * B : (j + 1) * B], dtype=np.float64, order="F")
    LT = scipy.linalg.lu_solve((lu, piv), T_j, overwrite_b=True, check_finite=False)
    fp_by_gloria[:, j, :] = (q[:, None] * LT).astype(np.float32)
    del T_j, LT
    if (j + 1) % 20 == 0 or j == G - 1:
        log(f"Baseline tensor solved for {j + 1}/{G} consuming countries")

fp_income = fp_by_gloria.sum(axis=0)
np.save(OUTPUT_DIR / "fp_by_gloria_baseline.npy", fp_by_gloria)
np.save(OUTPUT_DIR / "footprint_income_2023.npy", fp_income.astype(np.float32))
print(f"Household footprint total PDF: {fp_income.sum():.6e}")
print("Saved fp_by_gloria_baseline.npy and footprint_income_2023.npy.")

# Part 4. Final-product sector-category uncertainty.
# This is the final-product route. For each purchased product category C, Target is
# masked by purchased sector before Leontief propagation:
# fp_cat_C[s_prod,j,b] = q[s_prod] * solve(I-A, Target_C)[s_prod,j,b].

log("Loading population and sector classification")
pop_raw = pd.read_csv(FILES["population"], index_col=0)
mapping = pd.read_csv(FILES["country_mapping"])
pop = np.zeros((G, B), dtype=np.float64)
for gi in range(G):
    rows = mapping[mapping["GLORIA_Index"] == gi + 1]
    for _, row in rows.iterrows():
        iso3 = row["Population_ISO3"]
        if pd.notna(iso3) and iso3 != "Not in Population" and iso3 in pop_raw.columns:
            pop[gi] += pop_raw[iso3].values

sector_class = pd.read_excel(FILES["sector_class"], engine="openpyxl")
sector_class = sector_class.sort_values("Lfd_Nr") if "Lfd_Nr" in sector_class.columns else sector_class
sector_names = sector_class["Sector_names"].tolist()
assert len(sector_names) == S, f"sector_class.xlsx should have {S} rows"

sector_to_class = dict(zip(sector_class["Sector_names"], sector_class["class"]))
class_to_group = dict(zip(sector_class["class"], sector_class["5class"]))
class_order = list(dict.fromkeys(sector_class["class"].tolist()))
group_order = list(dict.fromkeys(sector_class["5class"].tolist()))
sector_class_array = np.array([sector_to_class[name] for name in sector_names])
purchased_sector = np.arange(R) % S

log("Building population-weighted global expenditure deciles")
exp_total = Target.reshape(R, G, B).sum(axis=0)
exp_pc = np.where(pop > 0, exp_total / pop, np.nan)
pop_f = pop.reshape(-1)
exp_f = exp_pc.reshape(-1)
valid = (pop_f > 0) & np.isfinite(exp_f)
order = np.argsort(np.where(valid, exp_f, np.inf), kind="mergesort")
Wfrac = np.zeros((G * B, 10), dtype=np.float64)
dec_pop = np.zeros(10, dtype=np.float64)
target_pop = pop_f[valid].sum() / 10.0
d = 0
remaining = target_pop
for u in order:
    if not valid[u]:
        break
    person_weight = pop_f[u]
    while person_weight > 1e-12 and d < 10:
        take = min(person_weight, remaining)
        Wfrac[u, d] += take / pop_f[u]
        dec_pop[d] += take
        person_weight -= take
        remaining -= take
        if remaining <= target_pop * 1e-12:
            d += 1
            remaining = target_pop

unbias = np.exp(-0.5 * sigma_matrix ** 2)
mc = (np.load(OUTPUT_DIR / "mc_factors_independent.npy") * unbias[None]).astype(np.float32)

def all_ratios_independent(mc_factors):
    out = np.zeros((mc_factors.shape[0], R), dtype=np.float32)
    for k in range(mc_factors.shape[0]):
        f_sector = mc_factors[k][r_index]
        sat_k = np.einsum("ls,sl->s", sat_by_lu, f_sector)
        out[k] = np.where(sat_base > 1e-30, sat_k / sat_base, 1.0).astype(np.float32)
    return out

log("Building independent ratio matrix")
AR = all_ratios_independent(mc)
del mc
gc.collect()

rows_detail = []
group_draws = {group: np.zeros((N_ITER, G, B), dtype=np.float32) for group in group_order}
group_base = {group: np.zeros((G, B), dtype=np.float64) for group in group_order}
reconstructed_total = np.zeros((G, B), dtype=np.float64)
solved_sum = np.zeros((R, G, B), dtype=np.float32)

def process_category(category, fp_cat):
    base2d = fp_cat.sum(axis=0).astype(np.float64)
    reconstructed_total[:] += base2d
    group = class_to_group[category]
    group_base[group] += base2d
    fp_flat = fp_cat.reshape(R, G * B)
    draws = (AR @ fp_flat).reshape(N_ITER, G, B).astype(np.float32)
    gini_draws = np.array([compute_gini(draws[k], pop) for k in range(N_ITER)])
    decile_fp = draws.reshape(N_ITER, -1) @ Wfrac
    top10 = decile_fp[:, 9] / decile_fp.sum(axis=1) * 100
    bottom50 = decile_fp[:, :5].sum(axis=1) / decile_fp.sum(axis=1) * 100
    gm, glo, ghi = ci(gini_draws)
    tm, tlo, thi = ci(top10)
    bm, blo, bhi = ci(bottom50)
    group_draws[group] += draws
    del draws
    rows_detail.append({
        "class": category,
        "5class": group,
        "cat_total_PDF": float(base2d.sum()),
        "gini_baseline": round(compute_gini(base2d, pop), 4),
        "gini_med_indep": round(gm, 4),
        "gini_lo95_indep": round(glo, 4),
        "gini_hi95_indep": round(ghi, 4),
        "top10_med_indep": round(tm, 2),
        "top10_lo95_indep": round(tlo, 2),
        "top10_hi95_indep": round(thi, 2),
        "bot50_med_indep": round(bm, 2),
        "bot50_lo95_indep": round(blo, 2),
        "bot50_hi95_indep": round(bhi, 2),
    })

skip_category = class_order[-1]
solve_categories = [category for category in class_order if category != skip_category]
for idx_category, category in enumerate(solve_categories, start=1):
    sector_idx = np.where(sector_class_array == category)[0]
    mask = np.isin(purchased_sector, sector_idx)
    log(f"Final-product category {idx_category}/{len(solve_categories)}: {category}")
    T_masked = np.zeros((R, G * B), dtype=np.float64, order="F")
    T_masked[mask] = Target[mask].astype(np.float64)
    LT = scipy.linalg.lu_solve((lu, piv), T_masked, overwrite_b=True, check_finite=False)
    del T_masked
    gc.collect()
    fp_cat = (q[:, None] * LT).astype(np.float32).reshape(R, G, B)
    del LT
    gc.collect()
    solved_sum += fp_cat
    process_category(category, fp_cat)
    del fp_cat
    gc.collect()

log(f"Final-product category by subtraction: {skip_category}")
fp_cat_last = fp_by_gloria - solved_sum
process_category(skip_category, fp_cat_last)
del fp_cat_last, solved_sum
gc.collect()

diff = np.abs(reconstructed_total - fp_income).max()
print(f"Reconstruction check max absolute difference: {diff:.3e}")

rows_group = []
for group in group_order:
    base2d = group_base[group]
    draws = group_draws[group]
    gini_draws = np.array([compute_gini(draws[k], pop) for k in range(N_ITER)])
    decile_fp = draws.reshape(N_ITER, -1) @ Wfrac
    top10 = decile_fp[:, 9] / decile_fp.sum(axis=1) * 100
    bottom50 = decile_fp[:, :5].sum(axis=1) / decile_fp.sum(axis=1) * 100
    gm, glo, ghi = ci(gini_draws)
    tm, tlo, thi = ci(top10)
    bm, blo, bhi = ci(bottom50)
    rows_group.append({
        "5class": group,
        "cat_total_PDF": float(base2d.sum()),
        "gini_baseline": round(compute_gini(base2d, pop), 4),
        "gini_med_indep": round(gm, 4),
        "gini_lo95_indep": round(glo, 4),
        "gini_hi95_indep": round(ghi, 4),
        "top10_med_indep": round(tm, 2),
        "top10_lo95_indep": round(tlo, 2),
        "top10_hi95_indep": round(thi, 2),
        "bot50_med_indep": round(bm, 2),
        "bot50_lo95_indep": round(blo, 2),
        "bot50_hi95_indep": round(bhi, 2),
    })

df_detail = pd.DataFrame(rows_detail)
df_group = pd.DataFrame(rows_group)
category_xlsx = OUTPUT_DIR / "uncertainty_category_results.xlsx"
with pd.ExcelWriter(category_xlsx, engine="openpyxl") as writer:
    df_detail.to_excel(writer, sheet_name="Sector_class_Gini_FP", index=False)
    df_group.to_excel(writer, sheet_name="Sector_5class_Gini_FP", index=False)

print("Saved final-product sector-category uncertainty:", category_xlsx)
print(f"Total runtime: {(time.time() - t_wall) / 60:.1f} min")
